# Démosaïçage et bruit : tracés de l'étude et protocole sur données réelles

Accompagne `REFERENCE_TECHNIQUE.md`, chapitre 11 (et eoqual-denoising, §10).

1. Résultats de l'étude (fichiers `docs/figures/joint_denoising/*.json`) : transfert du bruit par démosaïqueur (κ), règle sur le facteur C, qualité et temps de calcul, CPSNR en fonction de C pour Hamilton-Adams et GBTF, chaînes démosaïçage + débruitage.
2. Protocole « démosaïquer, stabiliser avec (a, b), NL-Bayes σ = C, inverser » sur une mosaïque : simulation par défaut, ou tes données (cellule de paramètres).

Dépendances : eoqual-demosaicing, eoqual-denoising, plotly, tifffile, Pillow (scikit-image et eoqual-metrics en simulation).

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

HERE = Path.cwd() if (Path.cwd() / "real_data_protocol.py").exists() else Path("examples")
sys.path.insert(0, str(HERE))
FIG = HERE.parent / "docs" / "figures" / "joint_denoising"
bench = json.loads((FIG / "demosaicing_benchmark.json").read_text())   # démosaïqueurs (§11.3, §11.7)
chains = json.loads((FIG / "results.json").read_text())                # chaînes (§11.6)
LEVELS = {"100": "fort (100 e⁻)", "400": "moyen (400 e⁻)", "1600": "faible (1600 e⁻)"}
noisy = {m: e for m, e in bench.items() if "noisy" in e}
def best(e, n):
    d = e["noisy"][n]; C = max(d, key=lambda c: d[c][0]); return float(C), d[C]
print(len(bench), "démosaïqueurs,", len(noisy), "évalués avec bruit")

## 1. Ce que le démosaïçage fait du bruit (§11.3)

κ = variance du bruit après / avant démosaïçage, en luminance Y et en chrominance U, V (champ uniforme). Un κ_Y > 1 concentre le bruit en luminance ; la chrominance, plus faible, devient corrélée (taches colorées).

In [ ]:
order = sorted(bench, key=lambda m: bench[m]["kappa"][0])
fig = go.Figure()
for i, name in enumerate(["κ_Y", "κ_U", "κ_V"]):
    fig.add_bar(x=order, y=[bench[m]["kappa"][i] for m in order], name=name)
fig.add_hline(y=1, line_dash="dot", annotation_text="bruit du capteur")
fig.update_layout(barmode="group", title="Gain de variance du bruit par démosaïqueur", yaxis_title="κ", height=450)
fig.show()

## 2. Facteur C sur σ : mesure et règle C = √κ_Y + 0.2 (§11.4)

Pour chaque démosaïqueur et niveau de bruit, C optimal mesuré (σ = C dans le domaine stabilisé, NL-Bayes couleur).

In [ ]:
fig = go.Figure()
for n, label in LEVELS.items():
    xs = [np.sqrt(e["kappa"][0]) for e in noisy.values()]
    ys = [best(e, n)[0] for e in noisy.values()]
    fig.add_scatter(x=xs, y=ys, mode="markers", name=label, text=list(noisy), hovertemplate="%{text}<br>√κ_Y %{x:.2f}<br>C* %{y}")
t = np.array([0.3, 1.7])
fig.add_scatter(x=t, y=t, mode="lines", line=dict(dash="dot", color="gray"), name="C = √κ_Y")
fig.add_scatter(x=t, y=t + 0.2, mode="lines", line=dict(dash="dash", color="black"), name="C = √κ_Y + 0.2 (règle)")
fig.update_layout(title="C optimal en fonction de √κ_Y", xaxis_title="√κ_Y", yaxis_title="C optimal", height=450)
fig.show()

## 3. Qualité et temps de calcul (§11.7)

CPSNR après débruitage (C optimal) en fonction du temps de démosaïçage d'une image de 4 Mpx (échelle log).

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=list(LEVELS.values()), shared_yaxes=False)
for k, n in enumerate(LEVELS, start=1):
    ms = list(noisy)
    fig.add_scatter(x=[noisy[m]["seconds_2048"] for m in ms], y=[best(noisy[m], n)[1][0] for m in ms],
                    mode="markers+text", text=ms, textposition="top center", textfont=dict(size=8),
                    marker=dict(color=["crimson" if m in ("ha", "gbtf") else "steelblue" for m in ms]),
                    showlegend=False, row=1, col=k)
    fig.update_xaxes(type="log", title_text="temps 4 Mpx (s)", row=1, col=k)
fig.update_yaxes(title_text="CPSNR (dB)", row=1, col=1)
fig.update_layout(title="Démosaïqueurs : qualité après débruitage vs temps (rouge : HA, GBTF)", height=480)
fig.show()

## 4. Hamilton-Adams et GBTF : CPSNR et ΔE en fonction de C

Repère vertical : C de la règle (1.36 pour HA, 1.75 pour GBTF).

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["CPSNR (dB)", "ΔE2000"])
colors = {"100": "#d62728", "400": "#ff7f0e", "1600": "#2ca02c"}
for m, dash in (("ha", "dot"), ("gbtf", "solid")):
    for n, label in LEVELS.items():
        d = bench[m]["noisy"][n]; Cs = sorted(d, key=float)
        for col, k in ((1, 0), (2, 1)):
            fig.add_scatter(x=[float(c) for c in Cs], y=[d[c][k] for c in Cs], mode="lines+markers",
                            line=dict(dash=dash, color=colors[n]), name=f"{m} — {label}", legendgroup=f"{m}{n}",
                            showlegend=(col == 1), row=1, col=col)
for m in ("ha", "gbtf"):
    fig.add_vline(x=np.sqrt(bench[m]["kappa"][0]) + 0.2, line_dash="dash", annotation_text=f"règle {m}")
fig.update_xaxes(title_text="C")
fig.update_layout(title="HA (pointillés) et GBTF (traits pleins)", height=450)
fig.show()

## 5. Chaînes démosaïçage + débruitage (§11.6)

In [ ]:
names = list(chains["cases"][0]["chains"])
fig = make_subplots(rows=1, cols=2, subplot_titles=["CPSNR (dB)", "ΔE2000"])
for n, label in ((100.0, LEVELS["100"]), (400.0, LEVELS["400"]), (1600.0, LEVELS["1600"])):
    cs = [c for c in chains["cases"] if c["full_e"] == n]
    for col, k in ((1, 0), (2, 1)):
        fig.add_bar(y=names, x=[np.mean([c["chains"][nm][k] for c in cs]) for nm in names], orientation="h",
                    name=label, legendgroup=label, showlegend=(col == 1), row=1, col=col)
fig.update_layout(barmode="group", height=750, title="Chaînes (moyenne des 4 zones)")
fig.show()

## 6. Protocole sur une mosaïque (§11.11)

**Paramètres** : laisser `RAW = None` pour une simulation depuis l'orthophoto ; sinon donner la mosaïque brute (DN), l'offset, DN_max, le modèle de bruit ``var(DN) = a_DN + b_DN · (DN − offset)`` et le motif. `CROP` limite le calcul (ligne, colonne, taille ; NL-Bayes ≈ 30 s par 4 Mpx et par valeur de C).

In [ ]:
IMAGE = "/Users/olivier/zPROJETS/IMAGES/capitole_2048_cog.tif"   # simulation : scène supposée sans bruit
RAW = None                     # données réelles : chemin .npy ou .tif de la mosaïque (DN)
OFFSET, DN_MAX = 64.0, 4095.0
A_DN, B_DN = None, None        # modèle var(DN) = a_DN + b_DN·(DN − offset)
PATTERN = "RGGB"
CROP = (400, 1040, 512)
FACTORS = {"ha": (1.0, 1.16, 1.36, 1.5), "gbtf": (1.25, 1.55, 1.75, 2.0)}

In [ ]:
import real_data_protocol as R

truth = None
if RAW is None:
    raw, truth, A_DN, B_DN = R.simulate_raw(IMAGE, electrons=400.0, crop=CROP)
    raw_offset = 64.0
else:
    import tifffile
    raw = np.load(RAW) if str(RAW).endswith(".npy") else tifffile.imread(RAW)
    r, c, n = CROP; r, c = r - r % 2, c - c % 2
    raw, raw_offset = raw[r:r + n, c:c + n], OFFSET
L = R.normalize_mosaic(raw, raw_offset, DN_MAX)
a, b = R.noise_model_to_normalized(A_DN, B_DN, raw_offset, DN_MAX)
print(f"a = {a:.3e}, b = {b:.3e} (luminance normalisée)")
metrics, results, rois = R.run_protocol(L, a, b, PATTERN, factors=FACTORS, truth=truth, out_dir=HERE / "output")

In [ ]:
keys = ["sigma_U", "sigma_V", "sigma_Y", "detail_ratio"] + (["cpsnr", "delta_e"] if truth is not None else [])
titles = {"sigma_U": "σ_U zones uniformes", "sigma_V": "σ_V zones uniformes", "sigma_Y": "σ_Y zones uniformes",
          "detail_ratio": "détails conservés (zones texturées)", "cpsnr": "CPSNR (dB)", "delta_e": "ΔE2000"}
fig = make_subplots(rows=2, cols=3, subplot_titles=[titles[k] for k in keys])
for m, cs in FACTORS.items():
    for i, k in enumerate(keys):
        fig.add_scatter(x=list(cs), y=[metrics["chains"][f"{m}|{float(C)}"][k] for C in cs], mode="lines+markers",
                        name=m, legendgroup=m, showlegend=(i == 0), row=i // 3 + 1, col=i % 3 + 1)
        fig.add_vline(x=R.rule_factor(m), line_dash="dot", row=i // 3 + 1, col=i % 3 + 1)
fig.update_xaxes(title_text="C")
fig.update_layout(height=650, title="Protocole : critères en fonction de C (pointillés : règle)")
fig.show()

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(HERE / "output" / "protocol_vignettes.png")))